# Build a model

Are the signals from `pbs-1`, `pbs-2`, and `pbs-3` sufficiently discrimatory of human breast cancer?


```
variation.update({
    'q_high': 0.95
})
```

## 0. Initializations

In [1]:
## 0. Initializations
# -- imports --
import anndata as ad
import gseapy as gp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns

import random

from collections import defaultdict

from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from statistics import median

from signals_in_the_noise.analysis.noise_phenotypes import classify_noise_subtypes
from signals_in_the_noise.preprocessing.gse161529 import GSE161529

In [2]:
# -- datasets --
gse = GSE161529()

## 1. Preprocessing Data

1. Subset data to samples that have a potential biological signal.
2. Determine the distribution of number of samples per subject - use to downsample for parity.
3. Determine the distribution of number of cells per subject - use to downsample for parity

In [3]:
initial = {
    'q_low': 0.25,
    'q_high': 0.75
}
variation = initial.copy()
variation.update({
    'q_high': 0.95
})
print(f"kwargs: {variation}")

kwargs: {'q_low': 0.25, 'q_high': 0.95}


In [4]:
# -- subset data --
subtype_to_adata = defaultdict(dict)
choices = ["pbs-1", "pbs-2", "pbs-3"]
for adata in gse.objects.values():
    adata_noise = adata[adata.obs["is_noise"] == 1].copy()
    classify_noise_subtypes(adata_noise, **variation)
    conditions = [
        adata_noise.obs["pbs-1"] == 1,
        adata_noise.obs["pbs-2"] == 1,
        adata_noise.obs["pbs-3"] == 1,
    ]
    adata_noise.obs["pbs"] = np.select(conditions, choices, default="none")
    adata_pbs_only = adata_noise[adata_noise.obs["pbs"] != "none"]
    subtype_to_adata[adata_pbs_only.uns['cancer_type']].update({
        adata_pbs_only.uns['title'] : adata_pbs_only
    })

In [5]:
subtype_to_adata.keys()

dict_keys(['Normal', 'BRCA1 pre-neoplastic', 'Triple negative tumour', 'Triple negative BRCA1 tumour', 'HER2+ tumour', 'PR+ tumour', 'ER+ tumour'])

In [6]:
def print_data_stats(adata_collection):
    results = []
    for subtype, adatas in adata_collection.items():
        num_subjects = len(adatas)
        num_samples = [adata.n_obs for adata in adatas.values()]
        print(f"{subtype} - {num_samples}")
        results.append((subtype, num_subjects, min(num_samples), median(num_samples)))
    
    return pd.DataFrame(results, columns=['Cancer sub-type', '# Subjects', 'Min # Samples', 'Median # Samples'])

print_data_stats(subtype_to_adata)

Normal - [26, 68, 9, 187, 412, 15, 10, 30, 11, 34, 18, 25, 34, 23, 9, 48, 38, 69, 28, 37, 22, 26, 169, 124]
BRCA1 pre-neoplastic - [38, 28, 60, 50]
Triple negative tumour - [24, 57, 13, 55]
Triple negative BRCA1 tumour - [39, 73, 152, 252]
HER2+ tumour - [46, 231, 42, 6, 73, 364]
PR+ tumour - [31]
ER+ tumour - [196, 152, 89, 66, 11, 45, 125, 42, 73, 76, 40, 36, 65, 29, 71, 26, 20, 2, 268, 233, 57, 73, 443, 11, 6]


,Cancer sub-type,# Subjects,Min # Samples,Median # Samples
0,Normal,24,9,29.0
1,BRCA1 pre-neoplastic,4,28,44.0
2,Triple negative tumour,4,13,39.5
3,Triple negative BRCA1 tumour,4,39,112.5
4,HER2+ tumour,6,6,59.5
5,PR+ tumour,1,31,31.0
6,ER+ tumour,25,2,65.0


### 1.a. Decisions

1. Exclude cancer sub-type `PR+ tumour` because there is only a single subject in the dataset.
2. Downsample subjects to `3` with a minimum of `50` samples.

### 1.b. Further subset data

Produces a dataset with the decisions.

In [7]:
def subset_data_for_modeling(require_min_subject=True, require_min_samples=True):
    results = {}
    for subtype, adatas in subtype_to_adata.items():
        if require_min_subject and subtype == 'PR+ tumour':
            # only has one subject, exclude
            continue
        valid_subjects_to_adata = defaultdict(dict)
        for adata in adatas.values():
            if require_min_samples and adata.n_obs < 50:
                # exclude subjects with fewer than 50 samples
                continue
            valid_subjects_to_adata.update({
                adata.uns['title']: adata,
            })
        results[subtype] = valid_subjects_to_adata
    return results
valid_adatas = subset_data_for_modeling()

In [8]:
print_data_stats(valid_adatas)

Normal - [68, 187, 412, 69, 169, 124]
BRCA1 pre-neoplastic - [60, 50]
Triple negative tumour - [57, 55]
Triple negative BRCA1 tumour - [73, 152, 252]
HER2+ tumour - [231, 73, 364]
ER+ tumour - [196, 152, 89, 66, 125, 73, 76, 65, 71, 268, 233, 57, 73, 443]


,Cancer sub-type,# Subjects,Min # Samples,Median # Samples
0,Normal,6,68,146.5
1,BRCA1 pre-neoplastic,2,50,55.0
2,Triple negative tumour,2,55,56.0
3,Triple negative BRCA1 tumour,3,73,152.0
4,HER2+ tumour,3,73,231.0
5,ER+ tumour,14,57,82.5


### 1.3. Utility to Generate Dataset for Modeling

Creates a list of dataframe from the AnnDatas that represents all cancer sub-types (except for `PR+ tumour`), with the following characteristics:

1. `3` subjects per sub-type, `6` sub-types = `18` total subjects
2. `1` dataframe per subject
3. Columns
   1. `has_cancer`
   2. `cancer_type`
   3. `cell_population`
   4. `menopause_status`
   5. `pbs-1`
   6. `pbs-2`
   7. `pbs-3`

Separating the subjects to a row will simplify using the leave-one-out strategy.

In [9]:
def generate_dataframe_for_modeling(downsample=True):
    results = []
    for adatas in valid_adatas.values():
        if downsample:
            subjects = np.random.choice(list(adatas.keys()), 3)
        else:
            subjects = adatas.keys()

        for subject in subjects:
            adata = adatas[subject]
            df = pd.DataFrame({
                'pbs-1': adata.obs['pbs-1'].astype(int),
                'pbs-2': adata.obs['pbs-2'].astype(int),
                'pbs-3': adata.obs['pbs-3'].astype(int),
            })
            df['has_cancer'] = (adata.uns['cancer_type'] != 'Normal')
            df['cancer_type'] = str(adata.uns['cancer_type'])
            df['cell_population'] = str(adata.uns['cell_population'])
            df['menopause_status'] = str(adata.uns['menopause_status'])
            
            results.append(df)
    return results

In [10]:
generate_dataframe_for_modeling()[0]

,pbs-1,pbs-2,pbs-3,has_cancer,cancer_type,cell_population,menopause_status
AACACACAGGCTATCT-1,0,1,0,False,Normal,Total,Post
AAGAACATCAGGTGTT-1,0,1,0,False,Normal,Total,Post
AAGCATCAGACGAAGA-1,0,1,0,False,Normal,Total,Post
AAGCGAGCAGGGTTGA-1,0,1,0,False,Normal,Total,Post
AAGCGAGGTATTGAGA-1,0,0,1,False,Normal,Total,Post
...,...,...,...,...,...,...,...
TTACCATGTTGTGTTG-1,0,1,0,False,Normal,Total,Post
TTCCTCTTCTCGTCAC-1,0,1,0,False,Normal,Total,Post
TTTACGTCATTGCCTC-1,0,1,0,False,Normal,Total,Post
TTTCCTCGTTCGTACA-1,0,1,0,False,Normal,Total,Post


## 2. Build the classifier

In [11]:
def logistic_regression_loo_classifier(dfs, *, by_subjects=True, as_dict=False, model=None, target='cancer_type'):
    loo = LeaveOneOut()
    y_true, y_pred = [], []
    
    for train_idx, test_idx in loo.split(dfs):
        train_df = pd.concat([dfs[i] for i in train_idx])
        # there will only ever be one in the test
        test_df = dfs[test_idx[0]]
    
        X_train = train_df[['pbs-1', 'pbs-2', 'pbs-3']].values
        y_train = train_df[target].values
    
        X_test = test_df[['pbs-1', 'pbs-2', 'pbs-3']].values
        y_test = test_df[target].values

        if not model:
            model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)
    
        if by_subjects:
            counts = np.unique(model.predict(X_test), return_counts=True)
            # return the majority vote for the subject
            y_pred.append(counts[0][np.argmax(counts[1])])
            y_true.append(test_df[target].iloc[0])
        else:
            y_pred.extend(model.predict(X_test))
            y_true.extend(y_test)
    
    return classification_report(y_true, y_pred, output_dict=as_dict)

# dfs = generate_dataframe_for_modeling()
# print(logistic_regression_loo_classifier(dfs))

In [12]:
# from sklearn.model_selection import LeaveOneOut
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import classification_report

# reports = []
# by_subject = False
# for run in range(10):
#     dfs = generate_dataframe_for_modeling()
#     report = logistic_regression_loo_classifier(dfs, as_dict=True)
#     reports.append(pd.DataFrame(report).T)

In [13]:
# avg_report = pd.concat(reports).groupby(level=0).mean()
# std_report = pd.concat(reports).groupby(level=0).std()

# print("Mean:")
# print(avg_report)
# print("\nStd:")
# print(std_report)

In [14]:
# dfs = generate_dataframe_for_modeling(downsample=False)
# print(logistic_regression_loo_classifier(dfs))

In [15]:
# dfs = generate_dataframe_for_modeling(downsample=False)
# print(logistic_regression_loo_classifier(dfs, by_subjects=False))

### 2.1. Explore other models

In [38]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import warnings
from sklearn.exceptions import UndefinedMetricWarning

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced'),
    'KNN': KNeighborsClassifier(n_neighbors=3, weights='distance'),
    'Naive Bayes': GaussianNB()
}

# with warnings.catch_warnings():
#     warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
#     n_runs = 100
#     for model_name, model in models.items():
#         reports = []
#         all_y_true, all_y_pred = [], []
        
#         for run in range(n_runs):
#             dfs = generate_dataframe_for_modeling(downsample=False)
#             report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model)
#             reports.append(pd.DataFrame(report).T)
    
#         avg_report = pd.concat(reports).groupby(level=0).mean()
    
#         print(f"{'='*25}")
#         print(f"Model: {model_name}")
#         print(f"{'-'*25}")
#         print(f"Mean: {avg_report}")
#         print()

In [17]:
# pd.concat(dfs).groupby('has_cancer')[['pbs-1','pbs-2','pbs-3']].mean()

### 2.2. Model ANY cancer

In [18]:
# with warnings.catch_warnings():
#     warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
#     n_runs = 100
#     for model_name, model in models.items():
#         reports = []
        
#         for run in range(n_runs):
#             dfs = generate_dataframe_for_modeling(downsample=False)
#             report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model, target="has_cancer")
#             reports.append(pd.DataFrame(report).T)
    
#         avg_report = pd.concat(reports).groupby(level=0).mean()
    
#         print(f"{'='*25}")
#         print(f"Model: {model_name}")
#         print(f"{'-'*25}")
#         print(f"Mean: {avg_report}")
#         print()

In [19]:
# pd.concat(dfs).groupby('has_cancer')[['pbs-1','pbs-2','pbs-3']].mean()

In [20]:
# with warnings.catch_warnings():
#     warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
#     n_runs = 100
#     for model_name, model in models.items():
#         reports = []
        
#         for run in range(n_runs):
#             dfs = generate_dataframe_for_modeling(downsample=False)
#             report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model, target="has_cancer")
#             reports.append(pd.DataFrame(report).T)
    
#         avg_report = pd.concat(reports).groupby(level=0).mean()
    
#         print(f"{'='*25}")
#         print(f"Model: {model_name}")
#         print(f"{'-'*25}")
#         print(f"Mean: {avg_report}")
#         print()

#### 2.3.1 Need to downsample to proceed

There are much more cancer than normal, this is skewing the results.

There are only `6` normal subjects.

In [21]:
dfs = generate_dataframe_for_modeling(downsample=False)

In [22]:
dfs[1]['cancer_type'].unique()[0]

'Normal'

In [23]:
dfs_cancer = []
dfs_normal = []

for df in dfs:
    if df['cancer_type'].unique()[0] == 'Normal':
        dfs_normal.append(df)
    else:
        dfs_cancer.append(df)

In [24]:
downsample_indices = random.sample(range(len(dfs_cancer)), len(dfs_normal)-1)
subset_cancer = [dfs_cancer[idx] for idx in downsample_indices]
dfs = dfs_normal.copy()
dfs.extend(subset_cancer)

balanced_df = pd.concat(dfs)
print(balanced_df['has_cancer'].value_counts())

has_cancer
False    1029
True      881
Name: count, dtype: int64


In [43]:
from sklearn.utils import resample

x = resample(dfs_cancer, n_samples=len(dfs_normal))
x.extend(dfs_normal)
balanced_df = pd.concat(x)
print(balanced_df['has_cancer'].value_counts())

has_cancer
False    1029
True      806
Name: count, dtype: int64


In [96]:
num_normal_cells = sum([df.count().unique()[0] for df in dfs_normal]) 

In [76]:
a = [cancer_df]

In [84]:
a

[[                    pbs-1  pbs-2  pbs-3  has_cancer cancer_type  \
  AAAGTAGTCGCGGATC-1      1      0      0        True  ER+ tumour   
  AACTCAGCACAGAGGT-1      0      0      1        True  ER+ tumour   
  AACTCCCGTTCTCATT-1      0      1      0        True  ER+ tumour   
  AAGACCTCATGGGACA-1      1      0      0        True  ER+ tumour   
  AAGCCGCAGTACGCGA-1      1      0      0        True  ER+ tumour   
  ...                   ...    ...    ...         ...         ...   
  TCGAGGCAGTAGCCGA-1      1      0      0        True  ER+ tumour   
  TGCCAAACAGGACGTA-1      0      0      1        True  ER+ tumour   
  TGGCGCATCCTTTACA-1      0      0      1        True  ER+ tumour   
  TTAACTCAGACATAAC-1      0      1      0        True  ER+ tumour   
  TTCGGTCCAGGTGCCT-1      0      1      0        True  ER+ tumour   
  
                     cell_population menopause_status  
  AAAGTAGTCGCGGATC-1           Total              nan  
  AACTCAGCACAGAGGT-1           Total              nan  
 

In [83]:
cancer_df[0]

,pbs-1,pbs-2,pbs-3,has_cancer,cancer_type,cell_population,menopause_status
AAAGTAGTCGCGGATC-1,1,0,0,True,ER+ tumour,Total,nan
AACTCAGCACAGAGGT-1,0,0,1,True,ER+ tumour,Total,nan
AACTCCCGTTCTCATT-1,0,1,0,True,ER+ tumour,Total,nan
AAGACCTCATGGGACA-1,1,0,0,True,ER+ tumour,Total,nan
AAGCCGCAGTACGCGA-1,1,0,0,True,ER+ tumour,Total,nan
...,...,...,...,...,...,...,...
TCGAGGCAGTAGCCGA-1,1,0,0,True,ER+ tumour,Total,nan
TGCCAAACAGGACGTA-1,0,0,1,True,ER+ tumour,Total,nan
TGGCGCATCCTTTACA-1,0,0,1,True,ER+ tumour,Total,nan
TTAACTCAGACATAAC-1,0,1,0,True,ER+ tumour,Total,nan


In [94]:
num_cancer_cells = 0
balance_cancer_dfs = []
queue_cancer_dfs = dfs_cancer.copy()
seen_cancer_dfs = []
while num_cancer_cells < num_normal_cells:
    while cancer_df in seen_cancer_dfs:
        cancer_df = random.sample(queue_cancer_dfs, 1)
    
    balance_cancer_dfs.append(cancer_df)
    num_cancer_cells += cancer_df[0].count().unique()[0]

num_cancer_cells

np.int64(1056)

In [161]:
def get_balanced_dataset(all_normals, all_abnormals, num_normal_subjects=6):
    # select the normal subjects first, its the smaller number of total subjects
    some_normals = random.sample(all_normals, k=min(num_normal_subjects, len(all_normals)))
    # total number of cells (samples) to use for downsampling the abnormal (cancer) subjects
    num_normal_cells = sum([df.count().unique()[0] for df in some_normals])

    abnormal_pool = all_abnormals.copy()
    some_abnormals = []
    num_abnormal_cells = 0
    # iterate over cancer subjects
    for _ in all_abnormals:
        abby = abnormal_pool.pop(random.randrange(len(abnormal_pool)))
        some_abnormals.append(abby)
        num_abnormal_cells += abby.count().unique()[0]
        if num_abnormal_cells >= num_normal_cells:
            break

    return some_normals, some_abnormals
    
# balanced_dfs_normal, balanced_dfs_cancer = get_balanced_dataset(dfs_normal, dfs_cancer)
# sum([df.count().unique()[0] for df in balanced_dfs_cancer]), sum([df.count().unique()[0] for df in balanced_dfs_normal])    

In [149]:
dfs = resample(dfs_cancer, n_samples=len(dfs_normal))
dfs.extend(dfs_normal)

In [155]:
type(dfs[0])

pandas.core.frame.DataFrame

In [156]:
type(balanced_dfs_normal[0])

pandas.core.frame.DataFrame

In [116]:
sum([df[0].count().unique()[0] for df in balance_cancer_dfs])

np.int64(1056)

In [95]:
seen_cancer_dfs

[]

In [89]:
len(balance_cancer_dfs)

16

In [57]:
[df.count().unique()[0] for df in dfs_normal]

[np.int64(68),
 np.int64(187),
 np.int64(412),
 np.int64(69),
 np.int64(169),
 np.int64(124)]

In [58]:
[df.count().unique()[0] for df in dfs_cancer]

[np.int64(60),
 np.int64(50),
 np.int64(57),
 np.int64(55),
 np.int64(73),
 np.int64(152),
 np.int64(252),
 np.int64(231),
 np.int64(73),
 np.int64(364),
 np.int64(196),
 np.int64(152),
 np.int64(89),
 np.int64(66),
 np.int64(125),
 np.int64(73),
 np.int64(76),
 np.int64(65),
 np.int64(71),
 np.int64(268),
 np.int64(233),
 np.int64(57),
 np.int64(73),
 np.int64(443)]

In [159]:
def report_has_cancer():
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
        n_runs = 100
        for model_name, model in models.items():
            reports = []
            for run in range(n_runs):
                # dfs = resample(dfs_cancer, n_samples=len(dfs_normal))
                # dfs.extend(dfs_normal)
                
                # downsample_indices = random.sample(range(len(dfs_cancer)), len(dfs_normal))
                # subset_cancer = [dfs_cancer[idx] for idx in downsample_indices]
                # dfs = dfs_normal.copy()
                # dfs.extend(subset_cancer)
                
                # num_cancer_cells = 0
                # balance_cancer_dfs = []
                # idx = 0
                # while num_cancer_cells < num_normal_cells:
                #     balance_cancer_dfs.append(dfs_cancer[idx])
                #     idx =+ 1
                #     num_cancer_cells += dfs_cancer[idx].count().unique()[0]

                # dfs = balance_cancer_dfs.copy()
                # dfs.extend(dfs_normal)
                balanced_dfs_normal, balanced_dfs_cancer = get_balanced_dataset(dfs_normal, dfs_cancer)
                dfs = balanced_dfs_cancer
                dfs.extend(balanced_dfs_normal)
                
                report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model, target="has_cancer")
                reports.append(pd.DataFrame(report).T)
        
            avg_report = pd.concat(reports).groupby(level=0).mean()
        
            print(f"{'='*25}")
            print(f"Model: {model_name}")
            print(f"{'-'*25}")
            print(f"Mean: {avg_report}")
            print()

report_has_cancer()

Model: Logistic Regression
-------------------------
Mean:               precision    recall  f1-score     support
False          0.566458  0.933407  0.704098   840.82000
True           0.858077  0.370055  0.512174   964.09000
accuracy       0.633170  0.633170  0.633170     0.63317
macro avg      0.712267  0.651731  0.608136  1804.91000
weighted avg   0.722948  0.633170  0.602490  1804.91000

Model: Decision Tree
-------------------------
Mean:               precision    recall  f1-score      support
False          0.567229  0.935093  0.705503   873.030000
True           0.862375  0.370212  0.515023   996.830000
accuracy       0.634651  0.634651  0.634651     0.634651
macro avg      0.714802  0.652653  0.610263  1869.860000
weighted avg   0.725177  0.634651  0.604813  1869.860000

Model: KNN
-------------------------
Mean:               precision    recall  f1-score      support
False          0.000000  0.000000  0.000000   850.070000
True           0.529386  1.000000  0.691899   961.2

In [162]:
def report_has_cancer():
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
        n_runs = 100
        for model_name, model in models.items():
            reports = []
            for run in range(n_runs):
                # dfs = resample(dfs_cancer, n_samples=len(dfs_normal))
                # dfs.extend(dfs_normal)
                
                # downsample_indices = random.sample(range(len(dfs_cancer)), len(dfs_normal))
                # subset_cancer = [dfs_cancer[idx] for idx in downsample_indices]
                # dfs = dfs_normal.copy()
                # dfs.extend(subset_cancer)
                
                # num_cancer_cells = 0
                # balance_cancer_dfs = []
                # idx = 0
                # while num_cancer_cells < num_normal_cells:
                #     balance_cancer_dfs.append(dfs_cancer[idx])
                #     idx =+ 1
                #     num_cancer_cells += dfs_cancer[idx].count().unique()[0]

                # dfs = balance_cancer_dfs.copy()
                # dfs.extend(dfs_normal)
                balanced_dfs_normal, balanced_dfs_cancer = get_balanced_dataset(dfs_normal, dfs_cancer)
                dfs = balanced_dfs_cancer
                dfs.extend(balanced_dfs_normal)
                
                report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model, target="has_cancer")
                reports.append(pd.DataFrame(report).T)
        
            avg_report = pd.concat(reports).groupby(level=0).mean()
        
            print(f"{'='*25}")
            print(f"Model: {model_name}")
            print(f"{'-'*25}")
            print(f"Mean: {avg_report}")
            print()

report_has_cancer()

Model: Logistic Regression
-------------------------
Mean:               precision    recall  f1-score      support
False          0.578533  0.934888  0.714269  1029.000000
True           0.855672  0.371237  0.514495  1125.250000
accuracy       0.641365  0.641365  0.641365     0.641365
macro avg      0.717103  0.653063  0.614382  2154.250000
weighted avg   0.723618  0.641365  0.610736  2154.250000

Model: Decision Tree
-------------------------
Mean:               precision    recall  f1-score      support
False          0.573531  0.934995  0.710423  1029.000000
True           0.860826  0.375248  0.519462  1156.620000
accuracy       0.639776  0.639776  0.639776     0.639776
macro avg      0.717179  0.655121  0.614943  2185.620000
weighted avg   0.725947  0.639776  0.610285  2185.620000

Model: KNN
-------------------------
Mean:               precision   recall  f1-score     support
False          0.000000  0.00000  0.000000  1029.00000
True           0.523840  1.00000  0.687309  1135.

In [ ]:
# ── DATA from avg_report ─────────────────────────────────────────────────────
support_false = avg_report.loc["False", "support"]
support_true  = avg_report.loc["True",  "support"]
recall_false  = avg_report.loc["False", "recall"]
recall_true   = avg_report.loc["True",  "recall"]
accuracy      = avg_report.loc["accuracy",     "f1-score"]
macro_f1      = avg_report.loc["macro avg",    "f1-score"]
f1_false      = avg_report.loc["False", "f1-score"]
f1_true       = avg_report.loc["True",  "f1-score"]
precision_false = avg_report.loc["False", "precision"]
precision_true  = avg_report.loc["True",  "precision"]

# reconstructed cm
tn = round(support_false * recall_false)
fp = round(support_false * (1 - recall_false))
fn = round(support_true  * (1 - recall_true))
tp = round(support_true  * recall_true)

cm       = np.array([[tn, fp], [fn, tp]])
support  = [support_false, support_true]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# ── DATA from avg_report ─────────────────────────────────────────────────────
support_false   = avg_report.loc["False",     "support"]
support_true    = avg_report.loc["True",      "support"]
recall_false    = avg_report.loc["False",     "recall"]
recall_true     = avg_report.loc["True",      "recall"]
precision_false = avg_report.loc["False",     "precision"]
precision_true  = avg_report.loc["True",      "precision"]
f1_false        = avg_report.loc["False",     "f1-score"]
f1_true         = avg_report.loc["True",      "f1-score"]
accuracy        = avg_report.loc["accuracy",  "f1-score"]
macro_f1        = avg_report.loc["macro avg", "f1-score"]

tn = round(support_false * recall_false)
fp = round(support_false * (1 - recall_false))
fn = round(support_true  * (1 - recall_true))
tp = round(support_true  * recall_true)

cm      = np.array([[tn, fp], [fn, tp]])
support = [support_false, support_true]
blue    = "#0072B2"
orange  = "#D55E00"

# ── FIGURE ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(8, 5))

ax_cm     = fig.add_axes([0.05, 0.10, 0.50, 0.80])
ax_metric = fig.add_axes([0.62, 0.10, 0.36, 0.80])

# ── CONFUSION MATRIX ─────────────────────────────────────────────────────────
classes = ["False\n(no cancer)", "True\n(cancer)"]

for i in range(2):
    for j in range(2):
        val = cm[i, j]
        pct = val / support[i] if support[i] > 0 else 0

        if i == j:
            color = blue
            alpha = 0.15 + pct * 0.75
        else:
            color = orange
            alpha = 0.10 + pct * 0.70

        rect = patches.FancyBboxPatch(
            (j + 0.06, 1 - i + 0.06), 0.88, 0.88,
            boxstyle="round,pad=0.02",
            facecolor=color, alpha=max(alpha, 0.08),
            linewidth=0
        )
        ax_cm.add_patch(rect)

        text_color = "white" if pct > 0.4 else "#111111"
        ax_cm.text(j + 0.5, 1 - i + 0.55, f"{int(val):,}",
                   ha="center", va="center",
                   fontsize=13, fontweight="500", color=text_color)
        ax_cm.text(j + 0.5, 1 - i + 0.30, f"{pct:.1%}",
                   ha="center", va="center",
                   fontsize=10, color=text_color, alpha=0.85)

ax_cm.set_xticks([0.5, 1.5])
ax_cm.set_xticklabels(classes, fontsize=10)
ax_cm.set_yticks([0.5, 1.5])
ax_cm.set_yticklabels(classes[::-1], fontsize=10)
ax_cm.set_xlim(0, 2)
ax_cm.set_ylim(0, 2)
ax_cm.set_xlabel("predicted label", fontsize=10)
ax_cm.set_ylabel("true label", fontsize=10)
ax_cm.spines[:].set_visible(False)
ax_cm.tick_params(length=0)

from matplotlib.patches import Patch
# legend_elements = [
#     Patch(facecolor=blue,   alpha=0.7, label="correct"),
#     Patch(facecolor=orange, alpha=0.7, label="misclassified"),
# ]
# ax_cm.legend(handles=legend_elements, loc="upper left",
#              frameon=False, fontsize=8,
#              bbox_to_anchor=(0, -0.08), ncol=2, handlelength=1)

# ── METRICS PANEL ────────────────────────────────────────────────────────────
ax_metric.axis("off")

ax_metric.text(0, 1.00, "accuracy",       transform=ax_metric.transAxes,
               fontsize=9, color="gray")
ax_metric.text(0, 0.92, f"{accuracy:.1%}", transform=ax_metric.transAxes,
               fontsize=20, fontweight="500")
ax_metric.text(0, 0.82, "macro F1",       transform=ax_metric.transAxes,
               fontsize=9, color="gray")
ax_metric.text(0, 0.74, f"{macro_f1:.3f}", transform=ax_metric.transAxes,
               fontsize=20, fontweight="500")

ax_metric.axhline(y=0.70, xmin=0, xmax=1,
                  color="#CCCCCC", linewidth=0.5)

ax_metric.text(0, 0.66, "recall / F1 by class", transform=ax_metric.transAxes,
               fontsize=9, color="gray")

metrics = [
    ("no cancer recall", recall_false, blue),
    ("cancer recall",    recall_true,  orange),
    ("no cancer F1",     f1_false,     blue),
    ("cancer F1",        f1_true,      orange),
]

bar_y = [0.58, 0.46, 0.31, 0.19]

for (label, val, color), y in zip(metrics, bar_y):
    ax_metric.text(0, y + 0.02, label, transform=ax_metric.transAxes,
                   fontsize=8, color="white")
    ax_metric.text(1, y + 0.02, f"{val*100:.2f}%", transform=ax_metric.transAxes,
                   fontsize=8, color="black", ha="right")
    ax_metric.add_patch(patches.FancyBboxPatch(
        (0, y), 1, 0.06,
        boxstyle="round,pad=0.005",
        transform=ax_metric.transAxes,
        facecolor="#EEEEEE", linewidth=0, clip_on=False
    ))
    if val > 0.001:
        ax_metric.add_patch(patches.FancyBboxPatch(
            (0, y), val, 0.06,
            boxstyle="round,pad=0.005",
            transform=ax_metric.transAxes,
            facecolor=color, alpha=0.8, linewidth=0, clip_on=False
        ))

ax_metric.axhline(y=0.14, xmin=0, xmax=1,
                  color="#CCCCCC", linewidth=0.5)

ax_metric.text(0, 0.08, f"no cancer  n={int(support_false):,}",
               transform=ax_metric.transAxes, fontsize=8, color="gray")
ax_metric.text(0, 0.00, f"cancer     n={int(support_true):,}",
               transform=ax_metric.transAxes, fontsize=8, color="gray")

legend_elements = [
    Patch(facecolor=blue,   alpha=0.8, label="correct"),
    Patch(facecolor=orange, alpha=0.8, label="misclassified"),
]
ax_metric.legend(handles=legend_elements,
                 loc="lower right",
                 bbox_to_anchor=(1, -0.01),
                 frameon=False, fontsize=8,
                 ncol=1, handlelength=1)

plt.suptitle("Naive Bayes — mean classification report", fontsize=11, y=0.97)
plt.savefig("images/umsi-expo-figure-4.png", dpi=300, bbox_inches="tight")
plt.show()

In [160]:
def report_cancer_sub_types():
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
        n_runs = 100
        for model_name, model in models.items():
            reports = []
            for run in range(n_runs):
                # dfs = resample(dfs_cancer, n_samples=len(dfs_normal))
                # dfs.extend(dfs_normal)

                # num_cancer_cells = 0
                # balance_cancer_dfs = []
                # idx = 0
                # while num_cancer_cells < num_normal_cells:
                #     balance_cancer_dfs.append(dfs_cancer[idx])
                #     idx =+ 1
                #     num_cancer_cells += dfs_cancer[idx].count().unique()[0]

                # dfs = balance_cancer_dfs.copy()
                # dfs.extend(dfs_normal)

                balanced_dfs_normal, balanced_dfs_cancer = get_balanced_dataset(dfs_normal, dfs_cancer)
                dfs = balanced_dfs_cancer
                dfs.extend(balanced_dfs_normal)
                
                # downsample_indices = random.sample(range(len(dfs_cancer)), len(dfs_normal)-1)
                # subset_cancer = [dfs_cancer[idx] for idx in downsample_indices]
                # dfs = dfs_normal.copy()
                # dfs.extend(subset_cancer)
                report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model)
                reports.append(pd.DataFrame(report).T)
        
            avg_report = pd.concat(reports).groupby(level=0).mean()
        
            print(f"{'='*25}")
            print(f"Model: {model_name}")
            print(f"{'-'*25}")
            print(f"Mean: {avg_report}")
            print()

report_cancer_sub_types()

Model: Logistic Regression
-------------------------
Mean:                               precision    recall  f1-score      support
BRCA1 pre-neoplastic           0.040413  0.064646  0.049471    67.777778
ER+ tumour                     0.531323  0.280587  0.348656   589.090000
HER2+ tumour                   0.029214  0.014524  0.019261   280.791045
Normal                         0.531498  0.833352  0.636403   863.580000
Triple negative BRCA1 tumour   0.053309  0.029158  0.037292   214.920635
Triple negative tumour         0.000000  0.000000  0.000000    65.040816
accuracy                       0.494105  0.494105  0.494105     0.494105
macro avg                      0.287000  0.296578  0.265227  1838.570000
weighted avg                   0.452680  0.494105  0.430630  1838.570000

Model: Decision Tree
-------------------------
Mean:                               precision    recall  f1-score      support
BRCA1 pre-neoplastic           0.026035  0.044755  0.032602    64.102564
ER+ tumour 

In [163]:
def report_cancer_sub_types():
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
        n_runs = 100
        for model_name, model in models.items():
            reports = []
            for run in range(n_runs):
                # dfs = resample(dfs_cancer, n_samples=len(dfs_normal))
                # dfs.extend(dfs_normal)

                # num_cancer_cells = 0
                # balance_cancer_dfs = []
                # idx = 0
                # while num_cancer_cells < num_normal_cells:
                #     balance_cancer_dfs.append(dfs_cancer[idx])
                #     idx =+ 1
                #     num_cancer_cells += dfs_cancer[idx].count().unique()[0]

                # dfs = balance_cancer_dfs.copy()
                # dfs.extend(dfs_normal)

                balanced_dfs_normal, balanced_dfs_cancer = get_balanced_dataset(dfs_normal, dfs_cancer)
                dfs = balanced_dfs_cancer
                dfs.extend(balanced_dfs_normal)
                
                # downsample_indices = random.sample(range(len(dfs_cancer)), len(dfs_normal)-1)
                # subset_cancer = [dfs_cancer[idx] for idx in downsample_indices]
                # dfs = dfs_normal.copy()
                # dfs.extend(subset_cancer)
                report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model)
                reports.append(pd.DataFrame(report).T)
        
            avg_report = pd.concat(reports).groupby(level=0).mean()
        
            print(f"{'='*25}")
            print(f"Model: {model_name}")
            print(f"{'-'*25}")
            print(f"Mean: {avg_report}")
            print()

report_cancer_sub_types()

Model: Logistic Regression
-------------------------
Mean:                               precision    recall  f1-score      support
BRCA1 pre-neoplastic           0.026097  0.043327  0.032173    63.617021
ER+ tumour                     0.551164  0.281323  0.359310   704.050000
HER2+ tumour                   0.052290  0.019519  0.027572   320.613333
Normal                         0.554622  0.784218  0.609663  1029.000000
Triple negative BRCA1 tumour   0.206564  0.125318  0.155502   210.216216
Triple negative tumour         0.000000  0.000000  0.000000    66.395833
accuracy                       0.482203  0.482203  0.482203     0.482203
macro avg                      0.308504  0.281780  0.264783  2190.840000
weighted avg                   0.486331  0.482203  0.433264  2190.840000

Model: Decision Tree
-------------------------
Mean:                               precision    recall  f1-score      support
BRCA1 pre-neoplastic           0.037129  0.058182  0.044981    65.272727
ER+ tumour 

In [179]:
def report_cancer_sub_types():
    # targets = ['HER2+ tumour', 'Triple negative tumour', 'Triple negative BRCA1 tumour']
    targets = ['ER+ tumour',]
    dfs_target_cancer = [df for df in dfs_cancer if df['cancer_type'].unique()[0] in targets]
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
        n_runs = 100
        for model_name, model in models.items():
            reports = []
            for run in range(n_runs):
                # dfs = resample(dfs_cancer, n_samples=len(dfs_normal))
                # dfs.extend(dfs_normal)

                # num_cancer_cells = 0
                # balance_cancer_dfs = []
                # idx = 0
                # while num_cancer_cells < num_normal_cells:
                #     balance_cancer_dfs.append(dfs_cancer[idx])
                #     idx =+ 1
                #     num_cancer_cells += dfs_cancer[idx].count().unique()[0]

                # dfs = balance_cancer_dfs.copy()
                # dfs.extend(dfs_normal)

                balanced_dfs_normal, balanced_dfs_cancer = get_balanced_dataset(dfs_normal, dfs_target_cancer, 6)
                dfs = balanced_dfs_cancer
                dfs.extend(balanced_dfs_normal)

                # dfs = dfs_target_cancer.copy()
                # dfs.extend(dfs_normal)
                
                # downsample_indices = random.sample(range(len(dfs_cancer)), len(dfs_normal)-1)
                # subset_cancer = [dfs_cancer[idx] for idx in downsample_indices]
                # dfs = dfs_normal.copy()
                # dfs.extend(subset_cancer)
                report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model)
                reports.append(pd.DataFrame(report).T)
        
            avg_report = pd.concat(reports).groupby(level=0).mean()
        
            print(f"{'='*25}")
            print(f"Model: {model_name}")
            print(f"{'-'*25}")
            print(f"Mean: {avg_report}")
            print()

report_cancer_sub_types()

Model: Logistic Regression
-------------------------
Mean:               precision    recall  f1-score     support
ER+ tumour     0.881373  0.450901  0.594543  1130.86000
Normal         0.609617  0.934888  0.737454  1029.00000
accuracy       0.681920  0.681920  0.681920     0.68192
macro avg      0.745495  0.692895  0.665998  2159.86000
weighted avg   0.752232  0.681920  0.663101  2159.86000

Model: Decision Tree
-------------------------
Mean:               precision    recall  f1-score      support
ER+ tumour     0.884777  0.460807  0.603880  1146.300000
Normal         0.610978  0.934888  0.738433  1029.000000
accuracy       0.685683  0.685683  0.685683     0.685683
macro avg      0.747877  0.697847  0.671156  2175.300000
weighted avg   0.755616  0.685683  0.668146  2175.300000

Model: KNN
-------------------------
Mean:               precision    recall  f1-score      support
ER+ tumour     0.522119  1.000000  0.685845  1127.750000
Normal         0.000000  0.000000  0.000000  1029.0

In [169]:
dfs_cancer[0]['cancer_type'].unique()[0]

'BRCA1 pre-neoplastic'

In [ ]:
targets = ['ER+ tumour',]
dfs_target_cancer = [df for df in dfs_cancer if df['cancer_type'].unique()[0] in targets]
len(dfs_target_cancer)